# Plate Reading Dataset

Merges the character annotations of the plate crops in `bangla-licence-plates` into one YOLO dataset with 103 classes (digits, series letters, `Metro`, and district words), removes training crops that duplicate validation or test crops, and publishes the result as `bangla-plate-reading`.

| Source | Format | Crops | Classes |
|---|---|---|---|
| `bd-lpr/Character Recognition` | Pascal VOC | 4,061 | 27, mapped to the names below |
| `bd-lpr-blpd/LPNR_Dataset` | YOLO | 7,217 | 105 |

**Class list**
- Two BLPD names for Cox's Bazar are merged, and `panchagarh` is capitalised.
- `Munshiganj` has two crops in total; those crops are dropped.
- Class names that differ only in capitals are different letters: `Ta` ট, `THA` ঠ, `Tha` থ, `Da` ড, `DA` দ, `Dha` ঢ.
- `classes.json` gives each class its Bangla text and kind (`digit`, `letter`, `word`, `district`), used to build the plate text from the detections.

**Setup:** add the `bangla-licence-plates` dataset as an input.

In [ ]:
%pip install -q imagehash

## Setup

In [ ]:
import json
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import imagehash
import numpy as np
import yaml
from PIL import Image
from tqdm.auto import tqdm

SOURCE = next(p for p in Path("/kaggle/input").rglob("bangla-licence-plates") if p.is_dir())
OUT = Path("/kaggle/working/plate-reading")
SPLITS = ("train", "val", "test")
IMAGE_EXT = {".jpg", ".jpeg", ".png"}
HAMMING_THRESHOLD = 2

if OUT.exists():
    shutil.rmtree(OUT)
for split in SPLITS:
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)

print("Source:", SOURCE)

## Classes

Order: digits, series letters, words, districts. The class id is the position in this list.

In [ ]:
DIGITS = {str(d): "০১২৩৪৫৬৭৮৯"[d] for d in range(10)}

LETTERS = {
    "A": "অ", "Ba": "ব", "Bha": "ভ", "Cha": "চ", "Chha": "ছ", "DA": "দ", "Da": "ড", "Dha": "ঢ",
    "E": "এ", "Ga": "গ", "Gha": "ঘ", "Ha": "হ", "Ja": "জ", "Jha": "ঝ", "Ka": "ক", "Kha": "খ",
    "La": "ল", "Ma": "ম", "Na": "ন", "Pa": "প", "Sa": "স", "Sha": "শ", "THA": "ঠ", "Ta": "ট",
    "Tha": "থ", "U": "উ",
}

WORDS = {"Metro": "মেট্রো", "Chatto": "চট্ট", "Raj": "রাজ"}

DISTRICTS = {
    "Bagerhat": "বাগেরহাট", "Bagura": "বগুড়া", "Bandarban": "বান্দরবান", "Barguna": "বরগুনা",
    "Barisal": "বরিশাল", "Bhola": "ভোলা", "Brahmanbaria": "ব্রাহ্মণবাড়িয়া", "Chandpur": "চাঁদপুর",
    "Chapainawabganj": "চাঁপাইনবাবগঞ্জ", "Chattogram": "চট্টগ্রাম", "Chuadanga": "চুয়াডাঙ্গা",
    "Coxs-Bazar": "কক্সবাজার", "Cumilla": "কুমিল্লা", "Dhaka": "ঢাকা", "Dinajpur": "দিনাজপুর",
    "Faridpur": "ফরিদপুর", "Feni": "ফেনী", "Gaibandha": "গাইবান্ধা", "Gazipur": "গাজীপুর",
    "Gopalganj": "গোপালগঞ্জ", "Habiganj": "হবিগঞ্জ", "Jamalpur": "জামালপুর", "Jessore": "যশোর",
    "Jhalokati": "ঝালকাঠি", "Jhenaidah": "ঝিনাইদহ", "Joypurhat": "জয়পুরহাট", "Khagrachari": "খাগড়াছড়ি",
    "Khulna": "খুলনা", "Kishoreganj": "কিশোরগঞ্জ", "Kurigram": "কুড়িগ্রাম", "Kustia": "কুষ্টিয়া",
    "Lakshmipur": "লক্ষ্মীপুর", "Lalmonirhat": "লালমনিরহাট", "Madaripur": "মাদারীপুর", "Magura": "মাগুরা",
    "Manikganj": "মানিকগঞ্জ", "Meherpur": "মেহেরপুর", "Moulvibazar": "মৌলভীবাজার", "Mymensingh": "ময়মনসিংহ",
    "Naogaon": "নওগাঁ", "Narail": "নড়াইল", "Narayanganj": "নারায়ণগঞ্জ", "Narsingdi": "নরসিংদী",
    "Natore": "নাটোর", "Netrokona": "নেত্রকোনা", "Nilphamari": "নীলফামারী", "Noakhali": "নোয়াখালী",
    "Nowabganj": "নবাবগঞ্জ", "Pabna": "পাবনা", "Panchagarh": "পঞ্চগড়", "Patuakhali": "পটুয়াখালী",
    "Pirojpur": "পিরোজপুর", "Rajbari": "রাজবাড়ী", "Rajshahi": "রাজশাহী", "Rangamati": "রাঙ্গামাটি",
    "Rangpur": "রংপুর", "Satkhira": "সাতক্ষীরা", "Shariatpur": "শরীয়তপুর", "Sherpur": "শেরপুর",
    "Sirajganj": "সিরাজগঞ্জ", "Sunamganj": "সুনামগঞ্জ", "Sylhet": "সিলেট", "Tangail": "টাঙ্গাইল",
    "Thakurgaon": "ঠাকুরগাঁও",
}

CLASSES = []
for kind, table in (("digit", DIGITS), ("letter", LETTERS), ("word", WORDS), ("district", DISTRICTS)):
    CLASSES += [{"name": name, "bangla": text, "kind": kind} for name, text in table.items()]
CLASS_ID = {c["name"]: i for i, c in enumerate(CLASSES)}

# source label -> class name; None drops every crop that contains the label
HOSSAIN_MAP = {
    **{d: d for d in DIGITS},
    "metro": "Metro", "dhaka": "Dhaka", "khulna": "Khulna", "chattogram": "Chattogram", "jashore": "Jessore",
    "ka": "Ka", "kha": "Kha", "ga": "Ga", "gha": "Gha", "ja": "Ja", "ba": "Ba", "ha": "Ha", "na": "Na",
    "ta": "Ta", "da": "Da", "ca": "Cha", "va": "Bha",
}
BLPD_MAP = {
    "Cox-s Bazar": "Coxs-Bazar", "Cox-s-Bazar": "Coxs-Bazar", "panchagarh": "Panchagarh", "Munshiganj": None,
}

assert len(CLASSES) == 103, len(CLASSES)
assert set(HOSSAIN_MAP.values()) <= set(CLASS_ID)
print(f"{len(CLASSES)} classes:", Counter(c["kind"] for c in CLASSES))

## Read annotations

Each crop keeps its source split and its character boxes in YOLO format (`class x_center y_center width height`, normalised). Labels missing from the maps stop the notebook, so no class is lost silently.

In [ ]:
def read_voc(image_path: Path, label_map: dict):
    root = ET.parse(image_path.with_suffix(".xml")).getroot()
    size = root.find("size")
    if size is not None and int(size.findtext("width", 0)) > 0:
        width, height = int(size.findtext("width")), int(size.findtext("height"))
    else:
        width, height = Image.open(image_path).size
    boxes = []
    for obj in root.iter("object"):
        b = obj.find("bndbox")
        x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, x2 = max(0.0, x1), min(float(width), x2)
        y1, y2 = max(0.0, y1), min(float(height), y2)
        if x2 > x1 and y2 > y1:
            boxes.append((obj.findtext("name"),
                          (x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height))
    return boxes


def read_yolo(image_path: Path, names: list):
    label = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    boxes = []
    if label.exists():
        for line in label.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                boxes.append((names[int(parts[0])], *map(float, parts[1:])))
    return boxes


blpd_root = SOURCE / "bd-lpr-blpd"
blpd_names = yaml.safe_load((blpd_root / "LPNR_data.yaml").read_text(encoding="utf-8"))["names"]

SOURCES = [
    # name, root, {source split: target split}, reader, reader argument, label map
    ("bd-lpr", SOURCE / "bd-lpr" / "Character Recognition",
     {"train": "train", "validation": "val", "test": "test"}, read_voc, None, HOSSAIN_MAP),
    ("blpd", blpd_root / "LPNR_Dataset",
     {"train": "train", "valid": "val", "test": "test"}, read_yolo, blpd_names, BLPD_MAP),
]

samples, dropped, unknown = [], Counter(), Counter()
for name, root, split_map, reader, argument, label_map in SOURCES:
    for source_split, split in split_map.items():
        images = [p for p in (root / source_split).rglob("*") if p.suffix.lower() in IMAGE_EXT]
        for image in tqdm(images, desc=f"{name}/{source_split}", leave=False):
            raw = reader(image, argument)
            # names not in the map are kept as they are and must be a class name
            mapped = [(label_map.get(l, l), *box) for l, *box in raw]
            if any(m is None for m, *_ in mapped):
                dropped[name] += 1
                continue
            unknown.update(m for m, *_ in mapped if m not in CLASS_ID)
            samples.append({"source": name, "split": split, "image": image,
                            "boxes": [(CLASS_ID[m], *box) for m, *box in mapped if m in CLASS_ID]})

assert not unknown, f"labels without a class: {dict(unknown)}"
print(f"{len(samples)} crops, {sum(len(s['boxes']) for s in samples)} character boxes; dropped crops: {dict(dropped)}")

## Remove duplicates across splits

Crops are compared by perceptual hash. Many plates look alike, so a training crop counts as a duplicate only when it is within `HAMMING_THRESHOLD` bits of a validation or test crop **and** has the same set of labels. Validation crops that duplicate test crops are removed the same way.

In [ ]:
def phash(path: Path) -> int:
    with Image.open(path) as image:
        return int(str(imagehash.phash(image.convert("L"))), 16)


for s in tqdm(samples, desc="hashing"):
    s["hash"] = phash(s["image"])
    s["signature"] = tuple(sorted(c for c, *_ in s["boxes"]))


def near_duplicates(candidates: list, references: list) -> set:
    groups = defaultdict(list)
    for r in references:
        groups[r["signature"]].append(r["hash"])
    groups = {k: np.array(v, dtype=np.uint64) for k, v in groups.items()}
    removed = set()
    for i, s in enumerate(candidates):
        ref = groups.get(s["signature"])
        if ref is None:
            continue
        xor = np.bitwise_xor(ref, np.uint64(s["hash"]))
        distance = np.unpackbits(xor.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
        if distance.min() <= HAMMING_THRESHOLD:
            removed.add(i)
    return removed


by_split = {split: [s for s in samples if s["split"] == split] for split in SPLITS}

drop_val = near_duplicates(by_split["val"], by_split["test"])
by_split["val"] = [s for i, s in enumerate(by_split["val"]) if i not in drop_val]

drop_train = near_duplicates(by_split["train"], by_split["val"] + by_split["test"])
by_split["train"] = [s for i, s in enumerate(by_split["train"]) if i not in drop_train]

print(f"Removed {len(drop_val)} validation and {len(drop_train)} training crops")

## Write the dataset

In [ ]:
summary, class_counts = Counter(), {split: Counter() for split in SPLITS}
for split, items in by_split.items():
    for s in tqdm(items, desc=split):
        stem = f"{s['source']}__{s['image'].stem}"
        shutil.copy2(s["image"], OUT / "images" / split / f"{stem}{s['image'].suffix.lower()}")
        lines = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in s["boxes"]]
        (OUT / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        summary[(split, s["source"], "images")] += 1
        summary[(split, s["source"], "boxes")] += len(s["boxes"])
        class_counts[split].update(c for c, *_ in s["boxes"])

(OUT / "data.yaml").write_text(
    "train: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {i}: {c['name']}\n" for i, c in enumerate(CLASSES))
)
(OUT / "classes.json").write_text(json.dumps(CLASSES, ensure_ascii=False, indent=2), encoding="utf-8")

rows = []
print(f"{'Split':6} {'Source':8} {'Crops':>7} {'Boxes':>7}")
for split in SPLITS:
    for name, *_ in SOURCES:
        images, boxes = (summary[(split, name, k)] for k in ("images", "boxes"))
        rows.append({"split": split, "source": name, "images": images, "boxes": boxes})
        print(f"{split:6} {name:8} {images:7d} {boxes:7d}")
    print(f"{split:6} {'total':8} {sum(summary[(split, n, 'images')] for n, *_ in SOURCES):7d}")

_ = (OUT / "summary.json").write_text(json.dumps({
    "rows": rows,
    "class_counts": {split: {CLASSES[c]["name"]: n for c, n in sorted(cnt.items())} for split, cnt in class_counts.items()},
    "removed_duplicates": {"val": len(drop_val), "train": len(drop_train)},
    "dropped_crops": dict(dropped),
    "hamming_threshold": HAMMING_THRESHOLD,
}, indent=2))

## Class balance

Boxes per class and split. Classes with fewer than 30 training boxes are listed; expect lower accuracy on them.

In [ ]:
import pandas as pd

balance = pd.DataFrame({split: [class_counts[split][i] for i in range(len(CLASSES))] for split in SPLITS},
                       index=[c["name"] for c in CLASSES])
balance["kind"] = [c["kind"] for c in CLASSES]
print(balance.groupby("kind")[list(SPLITS)].sum())
print("\nFewer than 30 training boxes:")
print(balance[balance["train"] < 30].sort_values("train").to_string())

## Samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(0)
fig, axes = plt.subplots(len(SPLITS), 5, figsize=(20, 3.5 * len(SPLITS)), squeeze=False)
for row, split in zip(axes, SPLITS):
    images = random.sample(sorted((OUT / "images" / split).iterdir()), 5)
    for ax, path in zip(row, images):
        image = Image.open(path).convert("RGB")
        width, height = image.size
        ax.imshow(image)
        for line in (OUT / "labels" / split / f"{path.stem}.txt").read_text().splitlines():
            c, x, y, w, h = line.split()
            x, y, w, h = map(float, (x, y, w, h))
            ax.add_patch(patches.Rectangle(((x - w / 2) * width, (y - h / 2) * height), w * width, h * height,
                                           fill=False, color="lime", lw=1.5))
            ax.text((x - w / 2) * width, (y - h / 2) * height, CLASSES[int(c)]["name"], color="yellow",
                    fontsize=7, va="bottom", backgroundcolor="black")
        ax.set_title(f"{split}: {path.name[:30]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()